# Découpage train / validation / test

**À exécuter une seule fois.** Ce notebook produit les trois partitions utilisées par toutes les expérimentations de l'étape 2. Le notebook d'entraînement les lit sans jamais les régénérer.

| Partition | Part | Rôle |
|---|---|---|
| `train.parquet` | 70 % | Apprentissage et ajustement des prétraitements |
| `val.parquet` | 15 % | Comparaison des modèles, optimisation des hyperparamètres |
| `test.parquet` | 15 % | Évaluation finale du seul modèle retenu |

`test` est relisible à volonté mais n'intervient dans **aucune décision** de modélisation, sous peine de perdre son indépendance. Si la préparation des données change, les partitions sont à régénérer et l'entraînement à relancer : les empreintes `data_version` et `split_version` calculées ici rendent cette dépendance visible dans chaque run MLflow.

## 1. Chargement et empreinte des données

In [ ]:
# Le projet est installe en mode editable par `uv sync`. Repli au cas ou le
# noyau selectionne ne serait pas celui du projet : on ajoute src/ au chemin.
try:
    import agritech  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    _racine = next(p for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / 'pyproject.toml').exists())
    sys.path.insert(0, str(_racine / 'src'))

import json

import pandas as pd

from agritech import decoupage
from agritech.config import (
    CONSOLIDATED_PARQUET_PATH,
    SPLIT_METADATA_PATH,
    SPLIT_PROPORTIONS,
    SPLIT_RANDOM_STATE,
    SPLIT_STRATIFY_ON,
    TARGET,
)

# Les trois partitions et leur fichier, dans l'ordre qui definit `split_version`
CHEMINS = decoupage.CHEMINS

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)

In [ ]:
# `charger_dataset` relit le consolide et calcule son empreinte par blocs d'un
# mega-octet. Il verifie au passage que l'index est unique : les controles de
# chevauchement raisonnent dessus, et seraient fausses sinon.
dataset, data_version = decoupage.charger_dataset()

print('Dataset consolide :', f'{len(dataset):,}', 'lignes x', dataset.shape[1], 'colonnes')
print('data_version      :', data_version)
print()
print('Colonnes :', list(dataset.columns))

## 2. Découpage stratifié

Le test est isolé d'abord, puis le reste séparé en entraînement et validation. La stratification par culture ne corrige aucun déséquilibre — `Crop` est uniformément répartie — mais garantit la présence des six cultures dans les trois jeux.

In [ ]:
# `decouper` isole d'abord `test`, puis recoupe le reste : `train_test_split`
# ne sait couper qu'en deux, d'ou la proportion recalculee pour `val`.
partitions = decoupage.decouper(dataset)

resume = pd.DataFrame({
    'lignes': {nom: len(d) for nom, d in partitions.items()},
    'part_%': {nom: round(100 * len(d) / len(dataset), 2) for nom, d in partitions.items()},
    'part_visee_%': {nom: 100 * p for nom, p in SPLIT_PROPORTIONS.items()},
})
resume.loc['total'] = [len(dataset), 100.0, 100.0]
resume

## 3. Contrôles d'intégrité

In [ ]:
# `controler` leve si une observation appartient a deux partitions, ou si une
# ligne du dataset n'appartient a aucune. Une fuite de `train` vers `test`
# fausserait l'evaluation finale sans que rien ne la signale.
repartition = decoupage.controler(partitions, dataset)

for nom, chiffres in repartition.items():
    print(f'OK    | {nom:<6} {chiffres["lignes"]:>9,} lignes   {chiffres["part_%"]:>5.2f} %')
print('OK    | aucun chevauchement entre partitions')
print('OK    | aucune ligne du dataset perdue')

### 3.1 Comparabilité des distributions

Un équilibre sur une variable ne dit rien des autres, ni la moyenne sur la dispersion : modalités, moyennes, écarts-types, queues de la cible et valeurs manquantes sont contrôlés séparément.

In [5]:
# Variables qualitatives : part de chaque modalite dans chaque partition.
# La liste est DEDUITE des types, pour qu'un changement de colonnes dans la
# preparation ne laisse pas ce controle porter sur des variables disparues.
qualitatives = dataset.select_dtypes(include=['object', 'str', 'bool']).columns.tolist()
print('Qualitatives controlees :', qualitatives)

ecarts = []
for colonne in qualitatives:
    reference = dataset[colonne].value_counts(normalize=True)
    for nom, partition in partitions.items():
        part = partition[colonne].value_counts(normalize=True).reindex(reference.index).fillna(0)
        ecarts.append({'variable': colonne, 'partition': nom,
                       'ecart_max_pp': round(100 * (part - reference).abs().max(), 3)})

ecarts = pd.DataFrame(ecarts).pivot(index='variable', columns='partition', values='ecart_max_pp')
print("Ecart maximal a la distribution d'ensemble, en points de pourcentage :")
ecarts

Qualitatives controlees : ['Region', 'Soil_Type', 'Crop', 'Fertilizer_Used', 'Irrigation_Used', 'Weather_Condition', 'crop_in_fao_reference', 'low_rain_no_irrigation']
Ecart maximal a la distribution d'ensemble, en points de pourcentage :


partition,test,train,val
variable,,,
Crop,0.000,0.000,0.000
Fertilizer_Used,0.071,0.009,0.029
Irrigation_Used,0.136,0.058,0.136
Region,0.235,0.047,0.132
Soil_Type,0.101,0.024,0.199
Weather_Condition,0.250,0.066,0.056
crop_in_fao_reference,0.000,0.000,0.000
low_rain_no_irrigation,0.001,0.023,0.110


In [6]:
# Variables quantitatives : position (moyenne) ET dispersion (ecart-type).
# Deux partitions peuvent partager la meme moyenne et differer par la dispersion.
# Liste deduite des types, cible comprise.
quantitatives = dataset.select_dtypes(include='number').columns.tolist()

lignes = []
for colonne in quantitatives:
    moy_ref, std_ref = dataset[colonne].mean(), dataset[colonne].std()
    moyennes = [partition[colonne].mean() for partition in partitions.values()]
    dispersions = [partition[colonne].std() for partition in partitions.values()]
    lignes.append({
        'variable': colonne,
        'moyenne': round(moy_ref, 4),
        # ecarts mesures avant arrondi, sinon on mesurerait l'arrondi lui-meme
        'ecart_max_moyenne': round(max(abs(m - moy_ref) for m in moyennes), 5),
        'ecart_type': round(std_ref, 4),
        'ecart_max_ecart_type': round(max(abs(s - std_ref) for s in dispersions), 5),
    })

print("Ecart maximal entre une partition et l'ensemble :")
pd.DataFrame(lignes).set_index('variable')

Ecart maximal entre une partition et l'ensemble :


,moyenne,ecart_max_moyenne,ecart_type,ecart_max_ecart_type
variable,,,,
Rainfall_mm,550.0778,0.53244,259.8043,0.57400
Temperature_Celsius,27.5061,0.01973,7.2205,0.01496
Yield_tons_per_hectare,4.6506,0.00135,1.6952,0.00424
crop_productivity_index,0.9977,0.00000,0.2839,0.00000
crop_yield_geo_cv,0.5312,0.00000,0.0878,0.00000
crop_yield_trend_decade,0.3172,0.00000,0.1022,0.00000


In [7]:
# Queues de la cible : un desequilibre des valeurs extremes ne se verrait
# ni dans la moyenne ni dans l'ecart-type
niveaux = [0.01, 0.25, 0.50, 0.75, 0.99]
quantiles = pd.DataFrame({nom: partition[TARGET].quantile(niveaux)
                          for nom, partition in partitions.items()})
quantiles['ensemble'] = dataset[TARGET].quantile(niveaux)

print('Quantiles de', TARGET, ':')
quantiles.round(4)

Quantiles de Yield_tons_per_hectare :


,train,val,test,ensemble
0.01,0.9971,0.9861,1.0085,0.9971
0.25,3.4191,3.4102,3.4249,3.4187
0.50,4.6532,4.6514,4.6487,4.6523
0.75,5.8802,5.8851,5.8710,5.8796
0.99,8.3121,8.3250,8.2997,8.3124


In [8]:
# Valeurs manquantes : la proportion doit etre stable d'une partition a l'autre
manquants = pd.DataFrame({
    nom: (100 * partition.isna().mean()).round(2)
    for nom, partition in partitions.items()
})
manquants['ensemble'] = (100 * dataset.isna().mean()).round(2)
manquants[manquants.sum(axis=1) > 0]

,train,val,test,ensemble
crop_productivity_index,33.34,33.34,33.33,33.34
crop_yield_geo_cv,33.34,33.34,33.33,33.34
crop_yield_trend_decade,33.34,33.34,33.33,33.34


> **Lecture.** On attend des écarts de l'ordre du hasard d'échantillonnage, pas des écarts nuls :
> sur 150 000 lignes, une proportion de 15 % fluctue déjà de ±0,09 point. Quelques dixièmes de
> point sont normaux, plusieurs points signaleraient une erreur.
>
> Le dernier tableau prime : une proportion de valeurs manquantes instable signifierait que les
> cultures hors référentiel FAO sont inégalement réparties, et les deux jeux de variables de
> l'étape 2 ne seraient plus comparables.

## 4. Les partitions sur le disque

**Ce notebook n'écrit aucune partition.** Elles sont produites une seule fois
par le script :

```bash
uv run python scripts/split_data.py
```

C'est lui qui porte les quatre cas, pour qu'aucune exécution ne puisse
invalider les runs MLflow en cours :

- **rien sur disque** → écriture ;
- **lot complet et à jour** (trois partitions *et* métadonnées, même
  `data_version`) → écriture ignorée, fichiers existants vérifiés ;
- **lot périmé**, généré pour un autre dataset → arrêt avec le diagnostic ;
  c'est le cas après toute modification de la préparation ;
- **lot incomplet** → arrêt : les fichiers présents peuvent provenir de
  découpages différents, et deviner serait pire que s'arrêter.

Les cellules ci-dessous lisent cet état et confrontent les fichiers au
découpage recalculé plus haut.

In [ ]:
# Lecture seule : `decider` ne fait que trancher, il n'ecrit rien lui-meme.
# Il leve avec un diagnostic si le lot est incomplet ou perime.
etat = decoupage.decider(data_version)
partitions_presentes = not etat.ecrire

print('Partitions sur le disque :', 'presentes' if partitions_presentes else 'absentes')
print('Etat                     :', etat.motif)

if not partitions_presentes:
    print()
    print('Les produire avec :')
    print('    uv run python scripts/split_data.py')

In [ ]:
if partitions_presentes:
    # Les metadonnees du disque font foi : elles seules disent de quel dataset
    # les partitions proviennent, et quels octets l'entrainement a lus.
    metadonnees = decoupage.metadonnees_du_disque()
    split_version = metadonnees['split_version']

    for nom, chemin in CHEMINS.items():
        print(f'{chemin.name:<16} {chemin.stat().st_size / 1024 ** 2:>7.1f} Mo')

    print()
    print('data_version  :', metadonnees['data_version'])
    print('split_version :', split_version)
else:
    metadonnees = None
    print('Aucune metadonnee a lire.')

In [ ]:
# Les metadonnees font partie du lot : des partitions sans leur `split_version`
# ne sont pas exploitables, puisque rien ne dirait de quel dataset elles
# proviennent ni quels octets l'entrainement a lus.
print(json.dumps(metadonnees, indent=2) if metadonnees else 'Aucune metadonnee.')

## 5. Relecture et vérification finale

In [ ]:
# Controle de sortie : les fichiers presents portent-ils bien les valeurs
# recalculees ici, et les metadonnees les decrivent-elles fidelement ?
# Le decoupage etant deterministe, la comparaison vaut meme sans reecriture :
# elle detecterait des fichiers issus d'un autre tirage.
if not partitions_presentes:
    print('Rien a verifier : les partitions n\'ont pas encore ete produites.')
else:
    effectifs = decoupage.relire_et_verifier(metadonnees, partitions)

    for nom, effectif in effectifs.items():
        print(f'OK    | {nom:<6} {effectif:>9,} lignes relues, '
              f'cible identique a 1e-12 pres')
    print('OK    | colonnes conformes aux metadonnees')
    print('OK    | split_version du disque identique a celui des metadonnees')

## À retenir

Partitions écrites et relues sans perte, contrôles finaux bloquants. **Ce notebook n'a plus à être exécuté**, sauf changement du dataset consolidé — qui imposerait de relancer entraînement et évaluation, et de traiter les runs antérieurs comme une autre version expérimentale.

`data_version` et `split_version` sont à reporter sur chaque run MLflow : deux runs d'empreintes différentes ne sont pas comparables.